# 10 · SVM for pump diagnostics

The practical part of lab 10. On operating records of centrifugal pumps you compare a linear model with kernel SVMs,
**tune C and γ on your own** with cross-validation, choose the decision threshold by the cost of errors and test the result once.
Then you try two other SVM tasks from the lecture: the one-class SVM trained without labelled faults and regression of the pump curve
with the ε-tube, including extrapolation. You have seen every setting you tune here in the educational part `svm-visual-en.ipynb`.

Short runnable examples use small illustrative data and variables prefixed with `demo_`. Run them in order;
they show the syntax of the whole workflow. Write your solution in the empty code cells below the tasks.
The test set is opened only in step 7. When you are done, run `Restart Kernel and Run All` to check that your solution runs in order.

## Environment setup and data loading

1. Open the notebook in Jupyter (the README describes the packages) or in Google Colab.
2. Run the example below. It imports every tool you will need and prepares small illustrative `demo_` data.
3. The prepared cell below it loads the data into the table `raw` directly from the course repository on GitHub
   (`10-SVM/data/pumps.csv`) and sets `SEED = 42`. It needs an internet connection.

### Example: imports and illustrative data

In [ ]:
from IPython.display import display                          # Display tables.
import matplotlib.pyplot as plt                              # Plots.
import numpy as np                                           # Numerical operations.
import pandas as pd                                          # Data and result tables.
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict, GridSearchCV  # Splitting, cross-validation and grid search.
from sklearn.pipeline import Pipeline                        # Chain scaling and a model.
from sklearn.preprocessing import StandardScaler             # Standardisation.
from sklearn.linear_model import LogisticRegression          # Logistic neuron from lecture 09.
from sklearn.svm import SVC, SVR, OneClassSVM                # SVM for classification, regression and one class.
from sklearn.metrics import confusion_matrix, roc_auc_score, mean_absolute_error  # Confusion matrix, ROC AUC, MAE.

demo_rng = np.random.default_rng(7)                          # Illustrative data, not the assignment data.
demo_X = pd.DataFrame({"a": demo_rng.normal(0, 1, 400), "b": demo_rng.normal(0, 10, 400)})  # Two features on different scales.
demo_y = ((demo_X["a"] ** 2 + (demo_X["b"] / 10) ** 2) > 1.6).astype(int)  # Class 1 outside a circle: no line separates it.
demo_X_dev, demo_X_test, demo_y_dev, demo_y_test = train_test_split(demo_X, demo_y, test_size=0.25, stratify=demo_y, random_state=7)  # A quarter for the test.
demo_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)  # Five folds with the same share of classes.
print(len(demo_X_dev), len(demo_X_test), "class 1 share:", round(demo_y.mean(), 3))  # Sizes and share.

**Prepared code:**

In [ ]:
SEED = 42                                                    # The same random split for everyone.
DATA_URL = ("https://raw.githubusercontent.com/CVUT-FS-12110/Artificial-Intelligence-and-Neural-Networks/"
            "master/10-SVM/data/pumps.csv")                  # The data file in the course repository on GitHub.
raw = pd.read_csv(DATA_URL)                                  # Download and read the table.
display(raw.head())                                          # The first five rows.

## Problem and data

Each row is one **operating record of a centrifugal pump** (all pumps are of the same type) with the result of an inspection `fault`
(1 = fault found). The data are synthetic; their description is in `data/README.md`.

| Column | Meaning | Typical value |
|---|---|---|
| `flow` | flow rate, the operating point | 20 to 80 m³/h |
| `pressure` | discharge pressure | 2 to 6 bar |
| `bearing_temp` | bearing temperature | 60 °C |
| `current` | motor current | 28 A |
| `vibration` | vibration velocity | 0.6 mm/s |
| `ambient_temp` | ambient temperature | 22 °C |

**Goal:** flag records that should be inspected. **Costs:** a missed fault (FN) costs **10** units (an unplanned stop),
a needless inspection (FP) costs **1** unit. The quality of a decision is measured by the **total cost** 10·FN + 1·FP.
(The cost is not called C here, so as not to mix it up with the SVM parameter C.)

## 1. Inspection (prepared)

**Output:** the share of faults, a table of feature ranges and two scatter plots. Look at where the faults lie relative to the
cloud of normal records in the plot of flow and pressure.

**Prepared code:**

In [ ]:
print("rows:", len(raw), "| share of faults:", round(raw["fault"].mean(), 3))  # Size and class share.
display(raw.drop(columns=["record_id", "fault"]).describe().T[["mean", "std", "min", "max"]].round(2))  # Ranges of the inputs.
fig, axs = plt.subplots(1, 2, figsize=(11, 4))               # Two plots side by side.
for ax, col in zip(axs, ["pressure", "bearing_temp"]):       # Pressure and bearing temperature against flow.
    for value, colour, label in ((0, "#0065BD", "normal"), (1, "#C2410C", "fault")):
        part = raw[raw["fault"] == value]
        ax.scatter(part["flow"], part[col], s=12, color=colour, label=label, alpha=0.7)
    ax.set(xlabel="flow", ylabel=col)
axs[0].legend()
plt.show()                                                   # Display the plots.

## 2. Features and a fixed split (prepared)

**Output:** `X` (six inputs) and `y` (`fault`); a development part `X_dev`, `y_dev` (675 records) and a test set `X_test`, `y_test`
(225 records) that stays closed until step 7; the folds `CV` (5-fold stratified cross-validation) used in every comparison.

**Prepared code:**

In [ ]:
FEATURES = ["flow", "pressure", "bearing_temp", "current", "vibration", "ambient_temp"]  # The six inputs.
X = raw[FEATURES]                                            # Inputs; record_id is not an input.
y = raw["fault"]                                             # Target class.
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)  # A quarter for the test, stratified.
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)  # The same five folds for every model.
print(f"dev {len(y_dev)} records, {int(y_dev.sum())} faults | test {len(y_test)} records, {int(y_test.sum())} faults")

## 3. Evaluation functions and a reference model (prepared)

**Output:** `cv_auc(model)` returns the mean ROC AUC of 5-fold cross-validation on the development data
(ROC AUC does not depend on the threshold, so it is used for tuning); `evaluate(y_true, y_pred)` returns the confusion-matrix counts,
precision, recall and the total cost. The reference is the logistic neuron from lab 09 with scaling.

**Prepared code:**

In [ ]:
def cv_auc(model, X_=None, y_=None, cv=None):
    # Mean ROC AUC over the folds; by default on the development data with CV.
    X_ = X_dev if X_ is None else X_
    y_ = y_dev if y_ is None else y_
    return cross_val_score(model, X_, y_, cv=CV if cv is None else cv, scoring="roc_auc").mean()


def evaluate(y_true, y_pred):
    # Counts, two metrics and the cost 10·FN + 1·FP.
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {"TN": tn, "FP": fp, "FN": fn, "TP": tp, "precision": tp / max(tp + fp, 1),
            "recall": tp / max(tp + fn, 1), "cost": 10 * fn + fp}


reference = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression())])  # Scaling, then a logistic neuron.
print("logistic neuron, CV ROC AUC:", round(cv_auc(reference), 3))  # About 0.5 = no better than chance.
print("cost of 'everything normal' on dev:", 10 * int(y_dev.sum()))  # Every fault missed.

## 4. Linear model versus kernels

**Goal / output:** a table `kernel_results` with the CV ROC AUC of four models on the development data:

1. linear SVM: `Pipeline` with `scale`: `StandardScaler()` → `model`: `SVC(kernel="linear")`;
2. polynomial kernel of degree 2: `SVC(kernel="poly", degree=2, coef0=1)` after scaling;
3. RBF kernel with default settings: `SVC()` after scaling;
4. the same RBF kernel **without** scaling.

**Use:** `Pipeline`, `StandardScaler`, `SVC`, `cv_auc`, `pd.Series` or `pd.DataFrame`.

<details><summary>Hint</summary>

The kernel goes in the `kernel` argument (`"linear"`, `"poly"`, `"rbf"`; the default is `"rbf"` with `C=1` and `gamma="scale"`).
`coef0=1` adds the constant r = 1 to the polynomial kernel (γ·x·z + 1)^d, so that it contains all products up to degree d.
Collect the results in a dictionary `{name: cv_auc(model)}` and turn it into `pd.Series(...)`.

</details>

**Think about:** Why is the linear model no better than chance (look at the plot in step 1)? Why does degree 2 already help?
How much does scaling matter, and why (educational part, section 5)?

### Example: kernels on the illustrative data

In [ ]:
demo_models = {                                              # Three kernels after scaling, one without.
    "linear": Pipeline([("scale", StandardScaler()), ("model", SVC(kernel="linear"))]),
    "poly 2": Pipeline([("scale", StandardScaler()), ("model", SVC(kernel="poly", degree=2, coef0=1))]),
    "RBF": Pipeline([("scale", StandardScaler()), ("model", SVC())]),
    "RBF unscaled": SVC(),
}
demo_results = pd.Series({name: cv_auc(model, demo_X_dev, demo_y_dev, demo_cv) for name, model in demo_models.items()}, name="CV ROC AUC")  # One number per model.
display(demo_results.round(3))                               # The circle needs a kernel and scaling.

**Your code:**

## 5. Tuning C and γ

**Goal / output:** a fitted `GridSearchCV` object `grid` for the RBF pipeline, a heatmap of its CV ROC AUC, and the final
`best_svm` (its `best_estimator_`) with the number of support vectors.

1. Pipeline `scale` → `model`: `SVC()`; grid `model__C` ∈ {0.1, 1, 10, 100}, `model__gamma` ∈ {0.01, 0.1, 1, 10};
   `cv=CV`, `scoring="roc_auc"`. Fit it on the development data.
2. Show the mean CV scores as a table C × γ (`pivot_table`) and as a heatmap.
3. If the best value lies **on the edge of the grid**, extend the grid in that direction (and around the best γ) and search again.
   Repeat while the best CV score still improves noticeably (by more than about 0.005). Store the final search in `grid`
   and its best model in `best_svm`.
4. Print the number of support vectors of `best_svm` (`best_svm[-1].n_support_`) compared with the number of training records.

**Use:** `GridSearchCV`, `.cv_results_`, `pd.DataFrame(...).pivot_table(...)`, `plt.imshow`, `.best_params_`, `.best_score_`, `.best_estimator_`.

<details><summary>Hint</summary>

Parameters of a pipeline step are named `step__parameter`, here `model__C` and `model__gamma`. The table:
`pd.DataFrame(grid.cv_results_).pivot_table(index="param_model__C", columns="param_model__gamma", values="mean_test_score")`.
`GridSearchCV` refits the best combination on all development data (`refit=True` by default).

</details>

**Think about:** In which direction did you extend the grid? Did both C and γ change? How does the best model relate to
the lecture slide on the joint choice of C and γ?

### Example: grid search on the illustrative data

In [ ]:
demo_pipe = Pipeline([("scale", StandardScaler()), ("model", SVC())])  # Scaling and an RBF SVM.
demo_grid = GridSearchCV(demo_pipe, {"model__C": [0.1, 1, 10], "model__gamma": [0.1, 1, 10]},  # Nine combinations.
                         cv=demo_cv, scoring="roc_auc").fit(demo_X_dev, demo_y_dev)  # Each one cross-validated.
demo_table = pd.DataFrame(demo_grid.cv_results_).pivot_table(index="param_model__C", columns="param_model__gamma", values="mean_test_score")  # C × γ table.
display(demo_table.round(3))
plt.imshow(demo_table.values, cmap="Blues", origin="lower")  # Heatmap; rows C, columns γ.
plt.xticks(range(3), demo_table.columns); plt.yticks(range(3), demo_table.index)
plt.xlabel("γ"); plt.ylabel("C"); plt.colorbar(); plt.show()
print(demo_grid.best_params_, round(demo_grid.best_score_, 3), "support vectors:", demo_grid.best_estimator_[-1].n_support_.sum())

**Your code:**

## 6. Threshold by the cost of errors

**Goal / output:** cross-validated scores `dev_scores` of the best settings on the development data, a plot of the total cost against
the threshold, the threshold `best_threshold` with the lowest cost, and a comparison with the threshold 0 (the default of `predict`).

1. `dev_scores = cross_val_predict(best_svm, X_dev, y_dev, cv=CV, method="decision_function")` – every record gets a score from a model
   that did not see it.
2. For thresholds from −2 to 1 (step 0.05) compute the cost of `(dev_scores >= threshold)`; plot it and find the minimum.
3. Print `evaluate` at the best threshold and at 0.

**Use:** `cross_val_predict`, `np.arange`, `evaluate`, `plt.plot`.

<details><summary>Hint</summary>

The SVM score is the signed distance-like value w·φ(x) + b, not a probability; `predict` uses the threshold 0. A negative threshold
flags more records as faults: fewer FN, more FP. With the cost ratio 10 : 1 the best threshold is usually below 0.

</details>

**Think about:** How many missed faults and how many needless inspections do the two thresholds give? Why is the threshold chosen on the
development data and not on the test set?

**Your code:**

## 7. Final test (once)

**Goal / output:** `test_scores` = `best_svm.decision_function(X_test)`, the ROC AUC on the test set and a table `test_result` with
`evaluate` at `best_threshold` and at 0, and the cost of “everything normal” on the test set. Nothing is tuned any more.

**Your code:**

## 8. One-class SVM without labelled faults (extension)

Steps 8 and 9 are an extension: do them in class if time allows, otherwise at home.

In practice, labelled faults are often missing; only records of normal operation exist. **Goal / output:** a table `oc_results`
with the validation ROC AUC of a one-class SVM for ν ∈ {0.02, 0.05, 0.1} and γ ∈ {0.03, 0.1, 0.3, 1}, the chosen settings,
and the test ROC AUC of the final one-class model `oc_model`.

1. Split the development data once more: `X_oc_train, X_oc_val, y_oc_train, y_oc_val` (30 % validation, stratified, `SEED`).
2. For each combination fit `Pipeline([("scale", StandardScaler()), ("model", OneClassSVM(nu=..., gamma=...))])`
   **only on the normal records** `X_oc_train[y_oc_train == 0]`; evaluate the ROC AUC on `X_oc_val` with the score
   `-model.decision_function(...)` (higher = more anomalous).
3. Refit the best combination on all normal development records; report the test ROC AUC, the share of normal test records flagged
   and the share of test faults found (`predict == -1`).

**Use:** `train_test_split`, `OneClassSVM`, `roc_auc_score`, `.decision_function`, `.predict`.

<details><summary>Hint</summary>

The labels of the validation part are used only to **evaluate** the settings, never to fit the model. `decision_function` of a one-class
SVM is positive inside the boundary, so its negative value is the anomaly score.

</details>

**Think about:** How far is the one-class model from the supervised SVM of step 7? When would you still prefer it?

**Your code:**

## 9. Pump curve: regression with the ε-tube and extrapolation (extension)

**Goal / output:** a table `svr_results` with the MAE of three SVR models inside and outside the training range, a plot of the three
curves, and a table `eps_results` with the number of support vectors and the MAE for ε ∈ {0.05, 0.2, 0.5}.

1. Take the **normal** development records (`y_dev == 0`). Inputs: `flow` only, target: `pressure`.
   Training range: flow < 65 m³/h; “new range”: flow ≥ 65 (the pump has never run there in the training data).
2. Fit three pipelines (scaling → `SVR`) with `C=10, epsilon=0.2`: linear kernel, polynomial `degree=2, coef0=1`, RBF `gamma=1`.
   Report the MAE in the training range and in the new range.
3. Plot the data and the three curves for flow from 20 to 80.
4. With the polynomial kernel change ε to 0.05, 0.2 and 0.5; report the number of support vectors and both MAE values.

**Use:** `SVR`, `mean_absolute_error`, `.support_`, `plt.plot`.

<details><summary>Hint</summary>

`SVR` needs a two-dimensional input: `data[["flow"]]`, not `data["flow"]`. The noise of the pressure around the curve has a standard
deviation of about 0.45 bar, so an MAE of about 0.27 is the best that any model can reach inside the data.

</details>

**Think about:** Which kernel extrapolates best and why (lecture slide on extrapolation)? Why does ε barely change the MAE
but strongly change the number of support vectors?

**Your code:**

## Summary

* A class that surrounds the other one (deviation on either side) needs a kernel; a linear model is no better than chance.
* C and γ are tuned together; a best value on the edge of the grid means extending the grid.
* The threshold on the SVM score is chosen by the cost of errors on the development data, then the test is opened once.
* The one-class SVM works without labelled faults, at the price of a weaker result.
* The RBF kernel does not extrapolate; a model with the right shape (here a parabola) does.